# Compare GNN architectures with their performance

In [15]:
#Importing necessary foundations

!pip install torch_geometric
import itertools
import torch
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
import time, copy
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, SAGEConv, GATConv


In [16]:
#Loading data
dataset = Planetoid(root="data/planetoid",name="Cora",transform = NormalizeFeatures())
data = dataset[0]
print(data)
print("Classes: ",dataset.num_classes,"\n features per node",dataset.num_node_features)
print("train/val/test:", data.train_mask.sum().item(),data.val_mask.sum().item(),data.test_mask.sum().item())
print("Undirected:", data.is_undirected(),"\n Isolated:", data.has_isolated_nodes(),"\n Self loops: ",data.has_self_loops())
print("Olass count", data.y.bincount().tolist())
print("train class count", data.y[data.train_mask].bincount().tolist())
src, dist = data.edge_index
print("edge homphily", (data.y[src] == data.y[dist]).float().mean().item())
deg = torch.bincount(dist,minlength = data.num_nodes)
print("degree min/midean/mid/max",deg.min().item(),deg.median().item(),round(deg.float().mean().item(),2),deg.max().item())

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
Classes:  7 
 features per node 1433
train/val/test: 140 500 1000
Undirected: True 
 Isolated: False 
 Self loops:  False
Olass count [351, 217, 418, 818, 426, 298, 180]
train class count [20, 20, 20, 20, 20, 20, 20]
edge homphily 0.8099659085273743
degree min/midean/mid/max 1 3 3.9 168


In [17]:
# MLP similar with minor modification adding dropout to make it robust and stops overfitting
#on model.eval() self.trainnig is false so preserved neurons , but on training mode we use dropout since it make some neurons off
# that help our model to stop memorizing and to stop overfitting
class MLP(nn.Module):
  def __init__(self,in_dim,hidden,out_dim,dropout = 0.5):
    super().__init__()
    self.l1 = nn.Linear(in_dim,hidden)
    self.l2 = nn.Linear(hidden,out_dim)
    self.dropout = dropout
  def forward(self,x,edge_index = None):
    x = F.relu(self.l1(F.dropout(x,self.dropout,self.training)))
    return self.l2(F.dropout(x,self.dropout,self.training))

In [18]:
#macro tester
#just f1 testing but it averages them after calculating inpendent f1
# since it is harmonic mean it helps ups to find the exact balance between the two
def f1(pred,true,num_classes):
  f1s = []
  for c in range(num_classes):
    tp = ((pred ==c ) & (true == c)).sum().item()
    fp = ((pred == c ) & (true != c)).sum().item()
    fn = ((pred != c) & (true == c)).sum().item()
    p = tp / (fp + tp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1s.append(2 * p * r / (p + r) if p + r else 0.0)
  return sum(f1s)/num_classes


In [19]:
'''
GNN
same as the previous layer but now used implemnted layer and make it versatilie for different GNN architecture
gat is the one unique due to its attention mechanism

ReLU on first layer only since we want to peresver our logits evaluation and training same as the first assignment
'''
class GNN(nn.Module):
  def __init__(self,kind,in_dim,hid,out_dim,layers=2,dropout=0.5, heads=8):
    super().__init__()
    self.dropout = dropout
    self.convs = nn.ModuleList()
    for i in range(layers):
      d_in  = in_dim  if i == 0 else hid
      last = (i == layers-1)
      d_out = out_dim if last else hid
      if kind == "gcn":
        conv = GCNConv(d_in,d_out)
      elif kind == "sage":
        conv = SAGEConv(d_in,d_out)
      else:
        conv = (GATConv(d_in, d_out,heads=1,dropout = dropout) if last
                else GATConv(d_in, d_out//heads,heads=heads,dropout = dropout) )
      self.convs.append(conv)

  def forward(self,x,edge_index,return_hidden = False):
    for conv in self.convs[:-1]:
      x = F.relu(conv(F.dropout(x,self.dropout,self.training),edge_index))
    h = x
    out = self.convs[-1](F.dropout(x,self.dropout,self.training),edge_index)
    return (out,h) if return_hidden else out

def build(kind,hid,layers = 2):
  d_in, d_out = dataset.num_node_features, dataset.num_classes
  return MLP(d_in,hid,d_out) if kind == "mlp" else GNN(kind,d_in,hid,d_out,layers)


def run(model,data, lr = 0.01, wd = 5e-4,epochs=300,edge_index = None, final = None):
  ei = data.edge_index if edge_index is None else edge_index

  optimizer = torch.optim.Adam(model.parameters(),lr=lr, weight_decay=wd)
  def accuracy(logits,y,mask):
    return (logits[mask].argmax(dim=1) ==y[mask]).float().mean().item()
  best_val_loss, best_state,t0 = float('inf'), None, time.time()
  for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    logits = model(data.x,ei)
    loss = F.cross_entropy(logits[data.train_mask],data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
      logits = model(data.x,ei)
      val_loss = F.cross_entropy(logits[data.val_mask],data.y[data.val_mask])
    if val_loss < best_val_loss:
        best_val_loss,best_state, = val_loss, copy.deepcopy(model.state_dict())
  train_time = time.time() - t0

  model.load_state_dict(best_state)
  model.eval()
  with torch.no_grad():
      pred = model(data.x,ei).argmax(1)
  res = {"val_acc":(pred[data.val_mask] == data.y[data.val_mask]).float().mean().item(),
         "time":train_time,
        "params": sum(p.numel() for p in model.parameters())
         }
  if final:
     res["test_acc"] = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
     res["test_f1"] = f1(data.y[data.test_mask],pred[data.test_mask],dataset.num_classes)
  return res

In [20]:
# Testing

for kind in ["mlp","gcn","sage","gat"]:
  torch.manual_seed(0)
  print(kind,run(build(kind,16),data,epochs=50,final= True))

mlp {'val_acc': 0.4339999854564667, 'time': 11.365302801132202, 'params': 23063, 'test_acc': 0.4560000002384186, 'test_f1': 0.46078094096804084}
gcn {'val_acc': 0.7239999771118164, 'time': 5.4763572216033936, 'params': 23063, 'test_acc': 0.7429999709129333, 'test_f1': 0.7483456572816133}
sage {'val_acc': 0.7739999890327454, 'time': 12.54130220413208, 'params': 46103, 'test_acc': 0.7839999794960022, 'test_f1': 0.7819127222202173}
gat {'val_acc': 0.7919999957084656, 'time': 6.671349763870239, 'params': 23109, 'test_acc': 0.7689999938011169, 'test_f1': 0.7692309937351}


Hyper parameter tunnig and fina_evalutaion pipeline

In [21]:
grid = list(itertools.product([16, 64], [0.01, 0.005]))
kinds = ["mlp", "gcn", "sage", "gat"]
best = {}

for kind in kinds:
  scores = []
  for hid, lr in grid:
    torch.manual_seed(0)
    r = run(build(kind,hid),data, lr = lr)
    scores.append((r['val_acc'],hid,lr))
    print(f"{kind:5s} hidden={hid:3d} lr={lr}  val_acc={r['val_acc']:.3f}")
  best[kind] = max(scores)
for k, v in best.items():
  print(f"{k:5s} {v}")

final = {}
for kind, (_,hid,lr) in best.items():
  final[kind] = []
  for seed in [0,1,2]:
    torch.manual_seed(seed)
    final[kind].append(run(build(kind, hid), data, lr=lr, final=True))

for kind,rs in final.items():
  a = [r['test_acc'] for r in rs]
  f = [r['test_f1'] for r in rs]
  print(f"{kind}: mean(a): {np.mean(a)} std(a): {np.std(a)} mean(f):{np.mean(f)} std(f): {np.std(f)}")



mlp   hidden= 16 lr=0.01  val_acc=0.598
mlp   hidden= 16 lr=0.005  val_acc=0.582
mlp   hidden= 64 lr=0.01  val_acc=0.598
mlp   hidden= 64 lr=0.005  val_acc=0.622
gcn   hidden= 16 lr=0.01  val_acc=0.794
gcn   hidden= 16 lr=0.005  val_acc=0.804
gcn   hidden= 64 lr=0.01  val_acc=0.806
gcn   hidden= 64 lr=0.005  val_acc=0.804
sage  hidden= 16 lr=0.01  val_acc=0.792
sage  hidden= 16 lr=0.005  val_acc=0.778
sage  hidden= 64 lr=0.01  val_acc=0.788
sage  hidden= 64 lr=0.005  val_acc=0.796
gat   hidden= 16 lr=0.01  val_acc=0.792
gat   hidden= 16 lr=0.005  val_acc=0.802
gat   hidden= 64 lr=0.01  val_acc=0.812
gat   hidden= 64 lr=0.005  val_acc=0.822
mlp   (0.621999979019165, 64, 0.005)
gcn   (0.8059999942779541, 64, 0.01)
sage  (0.7960000038146973, 64, 0.005)
gat   (0.8220000267028809, 64, 0.005)
mlp: mean(a): 0.5856666763623556 std(a): 0.0004714265495438534 mean(f):0.5677014776803063 std(f): 0.0009194851822797624
gcn: mean(a): 0.8240000009536743 std(a): 0.000816486068858174 mean(f):0.8120300637

In [23]:
# How removing different portion of edges affects performance
def drop_edges(edge_index,frac,seed):
  src,dist = edge_index
  und = edge_index[:,src < dist]
  g = torch.Generator().manual_seed(seed)
  n_keep = int(round((1 - frac) * und.shape[1]))
  idx = torch.randperm(und.shape[1],generator=g)[:n_keep].to(und.device)
  kept = und[:,idx]

  # for gluing and making same as edge_index while dropped
  return torch.cat([kept,kept.flip(0)],dim=1)
fracs = [0.0, 0.25, 0.5, 0.75, 1.0]
seeds = [0, 1]
kinds = ["mlp", "gcn", "sage", "gat"]

for f in fracs:
  print(f"Removed {f}: {drop_edges(data.edge_index,f,0).shape[1]} directed edges")

edge_res = {}
for kind in kinds:
  _, hid, lr = best[kind]
  for frac in fracs:
    accs = []
    for seed in seeds:
      torch.manual_seed(seed)
      ei = drop_edges(data.edge_index,frac,0)
      r = run(build(kind,hid),data,lr=lr,edge_index = ei,final = True)
      accs.append(r['test_acc'])
    edge_res[(kind,frac)] = (np.mean(accs), np.std(accs))
for kind  in kinds:
  print(f"Kind({kind}) mean and standard deviation respectively {', '.join([f'fraction:{f} ({edge_res[(kind,f)][0]:.3f} , {edge_res[(kind,f)][1]:.3f})' for f in fracs])}")

Removed 0.0: 10556 directed edges
Removed 0.25: 7916 directed edges
Removed 0.5: 5278 directed edges
Removed 0.75: 2640 directed edges
Removed 1.0: 0 directed edges
Kind(mlp) mean and standard deviation respectively fraction:0.0 (0.586 , 0.000), fraction:0.25 (0.586 , 0.000), fraction:0.5 (0.586 , 0.000), fraction:0.75 (0.586 , 0.000), fraction:1.0 (0.586 , 0.000)
Kind(gcn) mean and standard deviation respectively fraction:0.0 (0.824 , 0.000), fraction:0.25 (0.788 , 0.003), fraction:0.5 (0.750 , 0.008), fraction:0.75 (0.697 , 0.003), fraction:1.0 (0.576 , 0.012)
Kind(sage) mean and standard deviation respectively fraction:0.0 (0.814 , 0.002), fraction:0.25 (0.766 , 0.006), fraction:0.5 (0.720 , 0.003), fraction:0.75 (0.642 , 0.004), fraction:1.0 (0.581 , 0.004)
Kind(gat) mean and standard deviation respectively fraction:0.0 (0.827 , 0.002), fraction:0.25 (0.793 , 0.002), fraction:0.5 (0.749 , 0.002), fraction:0.75 (0.694 , 0.002), fraction:1.0 (0.579 , 0.004)


 ## Expirment Result

 ### General view
 1. From the expirment we can understand that MLP is not subjected to any kind of graph connection since it use pure weight rather than aggeragation of neighbours

 ### Differnt Settings in hyperparameters: Reports
1. MlP is stack at 0.6 even 64 neuron doing their best job on feature enginnering it is stuck since it doesn't see edges
2. GCN jums from 0.72 to 0.80 due to 64 neurons becuase cora has 1433 feature per node and putting that on 16 neuron is a massive bottleneck losing crictial information so upgrading to 64 increase 4times capicity to preserve word pattern.
3. While sage doens't show much difference wiht 0.76
4. GAT also shows mnior difference but with big score 0.82 -> Thsi shows attention mechanism works more rather than simply averaging like rnormal (GCN)  


we can conclude that  sage aggregation and flexibilty is overkilled since cora has small dataset gcn outperform it, Because Sage learns the genral patterns, but GCN assume the graph is fixed.And it is highly hompohelous graph for that reason.  global structural smoothing acts as an excellent low-pass filter, seamlessly averaging the features of highly similar, connected neighbors.

Unlike GCN gat and  sage show minor difference. The latter explicitly concatenates the target node's identity right next to the neighborhood message, it preserves the node's self-identity perfectly.while the former evaluates neighbour  before saving their info. Becuase it uses attention mechaninsm it only preserve the most important one reducting the discarded parts.

## How removing edges affect performance
1. We can see that MLP is consistent since it doesn't use the connection with neighbours
2. Generally it is shown with decrase number of fraction GNN Lose its ability to pinpoint the correct accuracy since it is dependent on neighbour aggeragation.
3. Both GCN and Sage falls fast about 24 - 25 % degradation from 0% degrading edges to 100% degrading edges
4. After edge removal all GNN arch will lean to MLP
-  MLP: 0.586
-  GCN: 0.576
- SAGE: 0.581
- GAT: 0.579 with minor % difference only

After seeing the result we can see GAT loses starting from 50% edge reduction while GCN beats it. The reason is that GAT use attenticon mechanisms with many parameters trying to caluclate attention cofficents over broken layouts. While GCN does simple normalization on what is left beating GAT and act as a regulazier.

# Conculsion
We can't simply say one model is better than other but depends on the structure of graph I can confidently say this GNN architecture is better than the other.  

1. For sturctured Graph with interlinked edges conencted: **GAT** is superiro due to its attention mechanisms.
2. For large graph that are not strutucred, which is broken:  **SAGE** is better
3. For small graph that are not strucuted and highly hompohelous graph: **GIN** is better